# Demo LIVE 2 · Senza guardare i polmoni

**CAF GIAS, Università degli Studi Link, modulo 2.4** · Michele Ferramola · slide «Senza guardare i polmoni»

Un buon AUROC non dice che cosa il modello ha imparato. Qui trova l'edema polmonare senza vedere i polmoni, perché la lastra fatta a letto si fa ai pazienti più gravi.

Senza vedere l'immagine, sapendo solo come è stata fatta la lastra,
un modello può riconoscere l'edema? Se non c'è nessun indizio, AUROC 0,5.


**I dati.** NIH ChestX-ray14 (NIH Clinical Center): 112.120 radiografie del torace di 30.805 pazienti.
Usiamo solo i metadati (età, sesso, proiezione e apparecchio), nessuna immagine. I reperti sono estratti in automatico dai referti, quindi contengono errori.
Uso libero citando Wang X. e colleghi, *ChestX-ray8*, IEEE CVPR 2017. Dati originali: https://nihcc.app.box.com/v/ChestXray-NIHCC

### Passo 1 · Di ogni radiografia teniamo età, sesso, proiezione e apparecchio

`a_letto` vale 1 se la lastra è fatta a letto (proiezione AP), 0 se in piedi (PA).
Larghezza, altezza e dimensione dei pixel identificano l'apparecchio.

**Atteso:** 112.120 radiografie, 30.805 pazienti, il 40% fatte a letto.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

DATI = "https://raw.githubusercontent.com/MicheleFerramola00/lezioni/caf-unilink-m2.4/dati/"
d = pd.read_parquet(DATI + "radiografie_nih_metadati.parquet")
print(f"{len(d):_} radiografie, {d.paziente.nunique():_} pazienti, fatte a letto: {d.a_letto.mean():.0%}".replace("_", "."))
d[["eta", "donna", "a_letto", "larghezza", "altezza", "edema"]].head()

112.120 radiografie, 30.805 pazienti, fatte a letto: 40%


,eta,donna,a_letto,larghezza,altezza,edema
0,57,0,0,2682,2749,0
1,58,0,0,2894,2729,0
2,58,0,0,2500,2048,0
3,80,0,0,2500,2048,0
4,74,1,0,2500,2048,0


### Passo 2 · Chi fa la lastra a letto

La lastra a letto si fa a chi non può alzarsi: i pazienti più gravi.

**Atteso:** 2.303 radiografie con edema su 112.120. Con edema il 4,5% delle lastre a letto, lo 0,4% di quelle in piedi.
Nel grafico: a letto l'88% delle radiografie con edema, il 35% di quelle senza reperti, il 40% di tutte.

In [ ]:
tab = pd.crosstab(d.a_letto.map({1: "a letto", 0: "in piedi"}), d.edema.map({1: "con edema", 0: "senza edema"}),
                  margins=True, margins_name="totale").rename_axis(index=None, columns=None)
print(tab.map(lambda x: f"{x:_}".replace("_", ".")), "\n")
pct = lambda x: f"{x:.1%}".replace(".", ",")
print(f"Con edema: il {pct(d[d.a_letto == 1].edema.mean())} delle lastre a letto, lo {pct(d[d.a_letto == 0].edema.mean())} di quelle in piedi.")

reperti = ["nessun_reperto", "versamento", "polmonite", "consolidamento", "edema"]
quota = pd.Series({r.replace("_", " "): d.loc[d[r] == 1, "a_letto"].mean() for r in reperti})
ax = quota.plot.barh(figsize=(8, 3.5), color=["#9a9892", "#2a78d6", "#2a78d6", "#2a78d6", "#D4A13A"])
for i, v in enumerate(quota):
    ax.text(v + 0.01, i, f"{v:.0%}", va="center")
ax.axvline(d.a_letto.mean(), color="#9a9892", ls="--", lw=1, label=f"tutte le radiografie: {d.a_letto.mean():.0%}")
ax.legend(loc="lower right", frameon=False)
ax.set_xlim(0, 1.05); ax.set_xlabel("quota di radiografie fatte a letto")
ax.set_title("La proiezione tradisce il reperto"); plt.tight_layout(); plt.show()

### Passo 3 · Un modello cerca l'edema senza vedere l'immagine

Il modello impara sul 70% dei pazienti e viene testato sui rimanenti: nessun paziente sta in entrambe le parti.

Prima proviamo il modello più semplice possibile, «classifichiamo sempre edema se la lastra è fatta a letto»

**Atteso:** AUROC 0,756 con la sola proiezione, 0,787 con età, sesso e apparecchio.

In [5]:
imp, ver = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0).split(d, groups=d.paziente))
d_imp, d_ver = d.iloc[imp], d.iloc[ver]
print(f"verifica: {len(d_ver):_} radiografie, {d_ver.edema.sum()} con edema".replace("_", "."))
print(f"solo la proiezione                     AUROC {roc_auc_score(d_ver.edema, d_ver.a_letto):.3f}")

colonne = ["a_letto", "eta", "donna", "larghezza", "altezza", "pixel_x", "pixel_y"]
modello = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(d_imp[colonne], d_imp.edema)
print(f"proiezione, età, sesso e apparecchio   AUROC {roc_auc_score(d_ver.edema, modello.predict_proba(d_ver[colonne])[:, 1]):.3f}")

verifica: 33.082 radiografie, 691 con edema
solo la proiezione                     AUROC 0.756
proiezione, età, sesso e apparecchio   AUROC 0.787


**Due modi di vedere il risultato.** Le barre mettono i nostri due AUROC accanto a quelli di due modelli che l'immagine la guardano, pubblicati sullo stesso dataset (Wang e colleghi 2017, CheXNet 2017). I loro pazienti di verifica non sono i nostri: il confronto è indicativo. Le curve ROC mostrano che la sola proiezione è un punto solo, e che il modello completo quasi coincide con lei.

**Atteso:** barre 0,756, 0,787, 0,835 e 0,888; nella curva ROC il punto «a letto» trova l'89% degli edemi con il 38% di falsi allarmi.

In [ ]:
auroc = pd.Series({
    "solo la proiezione": roc_auc_score(d_ver.edema, d_ver.a_letto),
    "proiezione, età, sesso e apparecchio": roc_auc_score(d_ver.edema, modello.predict_proba(d_ver[colonne])[:, 1]),
    "guarda l'immagine: Wang e colleghi, 2017": 0.835,
    "guarda l'immagine: CheXNet, 2017": 0.888,
}).iloc[::-1]
ax = auroc.plot.barh(figsize=(8, 3.5), color=["#c9c7c1", "#c9c7c1", "#2a78d6", "#D4A13A"])
for i, v in enumerate(auroc):
    ax.text(v + 0.01, i, f"{v:.3f}".replace(".", ","), va="center")
ax.axvline(0.5, color="#9a9892", ls="--", lw=1)
ax.text(0.51, 3.55, "0,5 = nessun indizio", fontsize=9, color="#6b6a66")
ax.set_ylim(-0.5, 4); ax.set_xlim(0, 1.05); ax.set_xlabel("AUROC sull'edema")
ax.set_title("Senza guardare i polmoni, quasi come chi li guarda")
plt.figtext(0.01, 0.01, "Barre grigie: valori pubblicati su ChestX-ray14, con pazienti di verifica diversi dai nostri: "
            "confronto indicativo.", fontsize=8, color="#6b6a66")
plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.show()

In [ ]:
from sklearn.metrics import roc_curve

punteggio = modello.predict_proba(d_ver[colonne])[:, 1]
fpr, tpr, _ = roc_curve(d_ver.edema, punteggio)
fp1 = d_ver.a_letto[d_ver.edema == 0].mean(); tp1 = d_ver.a_letto[d_ver.edema == 1].mean()
virgola = lambda x: f"{x:.3f}".replace(".", ",")
fig, ax = plt.subplots(figsize=(6, 5.2))
ax.plot([0, 1], [0, 1], color="#9a9892", ls="--", lw=1, label="nessun indizio: 0,5")
ax.plot(fpr, tpr, color="#2a78d6", label=f"proiezione, età, sesso e apparecchio: {virgola(roc_auc_score(d_ver.edema, punteggio))}")
ax.plot([0, fp1, 1], [0, tp1, 1], color="#D4A13A", label=f"solo la proiezione: {virgola(roc_auc_score(d_ver.edema, d_ver.a_letto))}")
ax.plot(fp1, tp1, "o", color="#D4A13A", ms=8)
ax.annotate(f"«a letto vuol dire edema»\ntrova il {tp1:.0%} degli edemi\ncon il {fp1:.0%} di falsi allarmi",
            (fp1, tp1), xytext=(fp1 + 0.08, tp1 - 0.3), arrowprops=dict(arrowstyle="-", color="#6b6a66"))
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_aspect("equal")
ax.set_xlabel("falsi allarmi: radiografie senza edema segnalate"); ax.set_ylabel("recall: edemi trovati")
ax.set_title("Le curve ROC: l'AUROC è l'area sotto la curva"); ax.legend(loc="lower right", frameon=False, title="AUROC")
plt.tight_layout(); plt.show()

### Passo 4 · Il risultato per sesso ed età, con quanti pazienti per gruppo

**Atteso:** donne 0,77, uomini 0,74; sopra i 70 anni 0,69, ma su soli 34 casi di edema.

In [3]:
gruppi = {"tutti": d_ver.eta >= 0, "donne": d_ver.donna == 1, "uomini": d_ver.donna == 0,
          "sotto i 30 anni": d_ver.eta < 30, "30-49 anni": d_ver.eta.between(30, 49),
          "50-69 anni": d_ver.eta.between(50, 69), "70 anni e oltre": d_ver.eta >= 70}
pd.DataFrame([(g, m.sum(), d_ver.edema[m].sum(), roc_auc_score(d_ver.edema[m], d_ver.a_letto[m]))
              for g, m in gruppi.items()],
             columns=["gruppo", "radiografie", "casi di edema", "AUROC della sola proiezione"]).round(2)

,gruppo,radiografie,casi di edema,AUROC della sola proiezione
0,tutti,33082,691,0.76
1,donne,14418,351,0.77
2,uomini,18664,340,0.74
3,sotto i 30 anni,5608,113,0.75
4,30-49 anni,11169,238,0.79
5,50-69 anni,13786,306,0.74
6,70 anni e oltre,2519,34,0.69


---

### Cosa abbiamo visto

1. **L'indizio sta nel modo in cui è stata fatta la lastra:** la lastra a letto porta con sé la gravità del paziente.
2. **La scorciatoia non vale uguale per tutti:** cambia fra sessi e fasce d'età.
3. **Con pochi casi un risultato per gruppo non si può giudicare:** va sempre letto accanto al numero di pazienti.

### Da chiedere in gara

> **Da quali ospedali, con quali apparecchi e protocolli? Su quanti ospedali è stato verificato?**